In [188]:
!pip install -U "langchain[google-genai]" langchain dotenv langgraph


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [126]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv
from pydantic import BaseModel, Field
load_dotenv()

model = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0.5, top_k=10, max_output_tokens=128)

In [72]:
print(type(model))

<class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>


In [152]:
response = model.invoke("What are cars?")

In [153]:
print(response)

content="Cars are **motor vehicles designed for transportation of people and goods**. They are a ubiquitous part of modern life, playing a crucial role in how we travel, work, and interact with the world.\n\nHere's a breakdown of what makes a car a car:\n\n**Key Characteristics:**\n\n*   **Self-Propelled:** Cars have their own internal power source, typically an engine, which allows them to move independently without being pulled or pushed.\n*   **Wheeled:** They have wheels that roll on the ground, enabling movement. Most cars have four wheels.\n*   **Road Vehicle:** Cars are designed to operate" additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--16d38fe1-3b34-4e4d-a96b-286622119f56-0' usage_metadata={'input_tokens': 5, 'output_tokens': 128, 'total_tokens': 133, 'input_toke

In [150]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [151]:
print(response.content)

Cars are **motor vehicles with wheels, typically four, designed to travel on roads and carry passengers.**

Here's a breakdown of what that means and some key aspects of cars:

**Core Components and Functionality:**

*   **Engine:** The heart of the car, it generates power to make the vehicle move. Historically, this has been an internal combustion engine (ICE) that burns fuel like gasoline or diesel. More recently, electric motors powered by batteries are becoming increasingly common.
*   **Wheels:** Usually four, these are what the car rolls on. They are connected to the drivetrain and allow for movement and


In [83]:
conversation = [
    SystemMessage("You are a helpful assistant that translates English to Hindi."),
    HumanMessage("Translate: I love programming."),
    AIMessage("Muzhe programming karna pasand hai."),
    HumanMessage("Translate: I love building applications.")
]

response = model.invoke(conversation)
print(response) 

content='Muzhe applications banana pasand hai.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--02ca2895-3058-49b4-b008-193714c454cd-0' usage_metadata={'input_tokens': 36, 'output_tokens': 9, 'total_tokens': 45, 'input_token_details': {'cache_read': 0}}


In [85]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

Parrots have colorful feathers for| a variety of fascinating reasons, and it's not just for show! The vibrant| hues play crucial roles in their survival and social lives. Here are the main reasons:

**1. Camouflage:**

*   **Blending In:** While many people associate parrots with bright, conspicuous colors, many species actually use their colors| to blend into their environment. In lush, green rainforests, for example, green feathers can make them incredibly difficult to spot by predators.
*   **Seasonal Changes:** Some parrots might have subtle color changes that help them blend in with different| foliage throughout the year.

**2||

In [86]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

content="Parrots have colorful feathers for a variety of fascinating reasons, and it's a complex interplay of **evolutionary pressures and social needs**. Here are the main drivers behind their vibrant plumage:\n\n**1. Communication and Identification:**\n\n* **Species Recognition:** Different parrot species have distinct color patterns, which are crucial for individuals to recognize their own kind. This helps them find mates, form flocks, and avoid interbreeding with other species. Imagine trying to find your flock in a dense rainforest if everyone looked the same!\n* **Individual Recognition:** Within a species, subtle variations in color or markings can help parrots identify familiar individuals, such" additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--377c4b32-165f-4d1b-

In [87]:
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)

(2, AIMessage(content="Quantum computing is a revolutionary new paradigm of computing that harnesses the principles of quantum mechanics to perform calculations. Unlike classical computers that use bits representing either 0 or 1, quantum computers use **qubits**.\n\nHere's a breakdown of what makes quantum computing so different and powerful:\n\n**The Core Concepts:**\n\n*   **Qubits (Quantum Bits):** This is the fundamental unit of information in quantum computing. Unlike a classical bit that can only be in a state of 0 or 1, a qubit can exist in a **superposition** of both 0 and 1 simultaneously. Imagine a spinning coin before", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--0867fd20-b71e-4742-82cf-caefe4dc4fa3-0', usage_metadata={'input_tokens': 6, 'output_tokens':

In [177]:
@tool
def get_weather(location: str) -> str:
    """Get the coordinates at a location."""
    return f"Rainy"  # Dummy coordinates for example

In [178]:
model_with_tools = model.bind_tools([get_weather])

In [179]:
response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


In [180]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# print messages
for message in messages:
    print(f"Message: {message}")

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

Message: {'role': 'user', 'content': "What's the weather in Boston?"}
Message: content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--5f0638f2-f2e4-4d69-8003-d488ea137353-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'b77574a2-de3a-4444-a977-5b968f75d360', 'type': 'tool_call'}] usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}}
Message: content='Rainy' name='get_weather' tool_call_id='b77574a2-de3a-4444-a977-5b968f75d360'
The weather in Boston is Rainy.


In [181]:
class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., required=True, description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., required=True, description="The movie's rating out of 10")
    description: str = Field(..., description="A brief description of the movie's plot")

/var/folders/46/5wp_ndn571nbzyh27y7141xr0000gn/T/ipykernel_35349/3995448775.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  title: str = Field(..., required=True, description="The title of the movie")
/var/folders/46/5wp_ndn571nbzyh27y7141xr0000gn/T/ipykernel_35349/3995448775.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  rating: float = Field(..., required=True, description="The movie's rating out of 10")


In [182]:
model_with_structure = model.with_structured_output(Movie, strict=False)
response = model_with_structure.invoke("Provide details about the movie The Matrix", config={"temperature": 0.7, "top_k": 5})
print(response)
print(type(response))

title='The Matrix' year=1999 director='The Wachowskis' rating=8.7 description='A computer hacker learns from mysterious rebels about the true nature of his reality and his role in the war against its creators.'
<class '__main__.Movie'>


In [183]:
model.get_name()

'ChatGoogleGenerativeAI'

In [184]:
model.get_prompts()

[]

In [186]:
!pip show langchain langgraph langchain-core

Name: langchain
Version: 1.0.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/bhushanshah/Documents/All-About-LLMs/AgenticAI/venv/lib/python3.13/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langgraph
Version: 1.0.2
Summary: Building stateful, multi-actor applications with LLMs
Home-page: 
Author: 
Author-email: 
License-Expression: MIT
Location: /Users/bhushanshah/Documents/All-About-LLMs/AgenticAI/venv/lib/python3.13/site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: langchain
---
Name: langchain-core
Version: 1.0.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /Users/bhushanshah/Documents/All-About-LLMs/AgenticAI/venv/lib/python3.13/site-packages
Requires: jsonpa

In [ ]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Successfully saved user info."

store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
response = agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})
print(response)
print("-----------------")
# Second session: get user info
response = agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
print(response)
# Here is the user info for user with ID "abc123":
# - Name: Foo
# - Age: 25
# - Email: foo@langchain.dev

{'messages': [HumanMessage(content='Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev', additional_kwargs={}, response_metadata={}, id='ccd59e2f-d78d-412e-9d27-c00c85fdb146'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_id": "abc123", "user_info": {"name": "Foo", "age": 25.0, "email": "foo@langchain.dev"}}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--eef5ef11-82d7-4197-8a40-82951875e11d-0', tool_calls=[{'name': 'save_user_info', 'args': {'user_id': 'abc123', 'user_info': {'name': 'Foo', 'age': 25.0, 'email': 'foo@langchain.dev'}}, 'id': '339cc702-1293-4b04-aa0e-991e6078832f', 'type': 'tool_call'}], usage_metadata={'input_tokens': 131, 'output_tokens': 51, 'total_tokens': 182, 'input_token_detai

In [203]:
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent

@tool
def log_tool_usage(tool_name: str, runtime: ToolRuntime):
    """Log the tool being used during this run."""
    runtime.state.setdefault("called_tools", []).append(tool_name)
    return f"Logged tool: {tool_name}"

@tool
def list_tools_used(runtime: ToolRuntime):
    """List tools called so far in this run."""
    return runtime.state.get("called_tools", [])

tools = [log_tool_usage, list_tools_used]
store = InMemoryStore()

agent = create_agent(model, tools, store=store)

# Call some tools
response = agent.invoke({
    "messages": [{"role": "user", "content": "Log that I used tool A"}]
})
print(response)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Now log that I used tool B"}]
})
print(response)

# Show which tools were used in this session
response = agent.invoke({
    "messages": [{"role": "user", "content": "List all tools I used so far"}]
})
print(response)


{'messages': [HumanMessage(content='Log that I used tool A', additional_kwargs={}, response_metadata={}, id='9b229059-0ef3-43cb-9f03-f78436a0b667'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'log_tool_usage', 'arguments': '{"tool_name": "A"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'}, id='lc_run--1264a37d-bb59-4fa7-a322-27bae55710e2-0', tool_calls=[{'name': 'log_tool_usage', 'args': {'tool_name': 'A'}, 'id': '03fb4961-373d-40a5-8cfe-c0c584e97cd1', 'type': 'tool_call'}], usage_metadata={'input_tokens': 86, 'output_tokens': 19, 'total_tokens': 105, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Logged tool: A', name='log_tool_usage', id='50a6ec41-79ee-4afa-a870-0d355e753dcb', tool_call_id='03fb4961-373d-40a5-8cfe-c0c584e97cd1'), AIMessage(content='', additional

In [204]:
for m in response['messages']:
    print(m)

content='List all tools I used so far' additional_kwargs={} response_metadata={} id='5aed8cec-9367-44e2-bdcd-56bd7f7b93b7'
content='' additional_kwargs={'function_call': {'name': 'list_tools_used', 'arguments': '{}'}} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--611190af-86a7-43fa-b8aa-316016df6ee2-0' tool_calls=[{'name': 'list_tools_used', 'args': {}, 'id': 'c66002bb-f60b-41f4-9c4e-555c43b86cfe', 'type': 'tool_call'}] usage_metadata={'input_tokens': 87, 'output_tokens': 12, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}}
content=[] name='list_tools_used' id='6b37abcc-2c7d-478d-8db9-05a4da09f8ef' tool_call_id='c66002bb-f60b-41f4-9c4e-555c43b86cfe'
content="I haven't used any tools yet." additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings':

In [201]:
response.keys()

dict_keys(['messages'])